# Tutorial: MAGMA Gene-Set and Gene Property Analysis

This tutorial demonstrates the MAGMA functions added to `cellink` for cell-type enrichment
analysis using GWAS summary statistics:

1. **MAGMA gene-set analysis (GSA)** — test whether top-scoring genes per cell type are enriched for GWAS signal.
2. **MAGMA gene property analysis (GPA)** — test the linear relationship between continuous per-gene scores and GWAS gene-level z-scores.

All functions are available at `cellink.tl.external`. MAGMA must be installed separately; download it from [https://ctg.cncr.nl/software/magma](https://ctg.cncr.nl/software/magma).

### What is a cell-type specificity score?

The central input to both analyses is a **specificity score**: for each gene and each cell type, a number capturing how selectively that gene is expressed in that cell type relative to all others. A gene with high specificity for, say, CD8 Naive T cells is expressed much more in that cell type than anywhere else; a gene with low specificity is expressed broadly across many cell types and isn't very informative about CD8 Naive biology specifically.

This tutorial computes specificity using the method from [Duncan et al. 2025](https://www.nature.com/articles/s41593-024-01834-w): for each gene, take its total expression across all cell types and compute what *fraction* of that total comes from each individual cell type. A gene's specificity scores across cell types therefore sum to 1 — a gene expressed equally everywhere has low specificity everywhere, while a gene expressed almost exclusively in one cell type has a specificity score near 1 there and near 0 elsewhere. The working assumption is that genes highly specific to a cell type are more likely to be the genes through which that cell type's biology contributes to disease risk — so testing whether GWAS signal concentrates near these genes is how we link a cell type to a trait.

> **Where this connects to LDSC:** these same specificity scores were originally developed for stratified LD-score regression (S-LDSC), a separate method for partitioning trait heritability across annotations using GWAS summary statistics and a population reference panel. `cellink`'s `preprocess_for_sldsc` function computes the scores (hence the name), but the scores themselves — and everything in this notebook — are used independently of LDSC; MAGMA's gene-set and gene-property tests consume them directly. If you want the full S-LDSC heritability-partitioning workflow instead, see the companion notebook `cell_level_ldsc_analysis_updates.ipynb`.

### GSA vs GPA — which one should I use?

Both tests start from the same specificity scores but ask the question differently:

- **GSA** thresholds each cell type's scores to a top-N% gene set (binary: in the set or not) and tests whether that set is enriched for GWAS signal. Simple to interpret, mirrors a standard binary enrichment test, and a good default — especially if you want results comparable to a published top-10%-style analysis.
- **GPA** uses the full continuous score for every gene (no thresholding) and tests the linear relationship between score and GWAS gene-level association. It avoids the arbitrary top-N% cutoff and can be more powerful when specificity varies smoothly across genes rather than splitting cleanly into "specific" vs "not."

If you're unsure, start with GSA for its simplicity; switch to GPA if a hard top-N% cutoff feels like it's throwing away real signal in your data.

### Required inputs

| Input | Description |
|---|---|
| `specificity_df` | Specific genes × cell-types DataFrame of per-gene scores (e.g. from `preprocess_for_sldsc`) |
| Gene coordinate file | Tab-separated file mapping genes to chr/start/end positions |
| GWAS SNP location file (for MAGMA Step I) | SNP/CHR/BP columns |
| MAGMA gene location file | Fetched from Ensembl via `get_magma_gene_loc` |
| PLINK bfile (for MAGMA Step II) | LD reference panel, e.g. 1000 Genomes EUR |
| GWAS p-value file (for MAGMA Step II) | SNP + P columns |
| `.genes.raw` file (for MAGMA Step III) | Output of Step II; can be pre-computed |


## Environment Setup

In [1]:
import requests, zipfile, io, os, stat

# Download MAGMA v1.10 (Linux, 64-bit, static linking)
# Official source / full download table: https://cncr.nl/research/magma/
url = "https://vu.data.surf.nl/index.php/s/lxDgt2dNdNr6DYt/download"

r = requests.get(url)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall("magma_bin")

# Make all extracted files executable
for root, dirs, files in os.walk("magma_bin"):
    for file in files:
        fpath = os.path.join(root, file)
        st = os.stat(fpath)
        os.chmod(fpath, st.st_mode | stat.S_IEXEC)


In [2]:
import cellink
import numpy as np
import pandas as pd
from pathlib import Path
import os

from cellink._core import DAnn, GAnn
from cellink.resources import get_dummy_onek1k, get_gwas_catalog_study_summary_stats
from cellink.tl.external import (
    preprocess_for_sldsc,
    generate_sldsc_genesets,
    scores_to_gmt,
    scores_to_covar,
    genesets_dir_to_entrez_gmt,
    run_magma_annotate,
    get_magma_gene_loc,
    run_magma_gene_analysis,
    run_magma_gsa,
    run_magma_gpa,
)
from cellink.resources import get_1000genomes_plink_files

# Path to the downloaded MAGMA binary (extracted by the cell above)
MAGMA_BIN = os.path.join("magma_bin", "magma")

# MAGMA gene location file — fetched from Ensembl and cached in ~/cellink_data/magma/
MAGMA_GENE_LOC = get_magma_gene_loc(genome_build="GRCh37")

# 1000G PLINK files — downloaded and cached in ~/cellink_data/1000genomes_plink_EUR/
_plink_path, _plink_prefix = get_1000genomes_plink_files(
    config_path=str(Path(cellink.__file__).parent / "resources" / "config" / "1000genomes.yaml"),
    population="EUR",
)
# Use chr22 as the LD reference for the tutorial (smallest autosome; use all chroms in production)
G1000_BFILE = str(_plink_path / f"{_plink_prefix}22")

# Cell type for single-cell examples
CELL_TYPE = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"


/lustre/groups/ml01/workspace/lucas.arnoldt/cellink/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-07-06 15:33:43,808] INFO:cellink.tl.external._sldsc_utils: Using cached MAGMA gene location file: /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt


[2026-07-06 15:33:43,835] INFO:root: /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G_Phase3_plinkfiles.tgz already exists


[2026-07-06 15:33:43,835] WARNING:root: No checksum provided, skipping verification


## Load and Prepare Data

We load the real OneK1K dataset and compute cell-type specificity scores as described above. (If you've already run the companion LDSC notebook, these preprocessing steps are identical — only the downstream analysis differs.)


In [3]:
dd = get_dummy_onek1k()
print(f"Dataset shape: {dd.shape}")
dd.C.obs[DAnn.donor] = dd.C.obs[original_donor_col]


[2026-07-06 15:33:43,909] INFO:root: /home/icb/lucas.arnoldt/cellink_data/dummy_onek1k/dummy_onek1k.dd.h5 already exists


[2026-07-06 15:33:43,914] INFO:root: Veryifying checksum


[2026-07-06 15:33:52,021] INFO:root: Loaded dummy OneK1K dataset: (100, 146939, 125366, 34073)


Dataset shape: (100, 146939, 125366, 34073)


/tmp/ipykernel_609401/2814731171.py:3: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  dd.C.obs[DAnn.donor] = dd.C.obs[original_donor_col]


In [4]:
# The dummy dataset is already small, but this shows the subsampling pattern
# you'd use on the full OneK1K dataset (~1.25M cells) for per-cell-type
# specificity estimation.
np.random.seed(0)
n_cells_keep = 30_000
cell_idx = np.sort(np.random.choice(dd.shape[2], min(n_cells_keep, dd.shape[2]), replace=False))
dd = dd[:, :, cell_idx, :].copy()
print(f"Subsampled to {dd.shape[2]:,} cells")


Subsampled to 30,000 cells


In [5]:
dd.C.var["gene"] = dd.C.var_names
adata = dd.C.copy()

adata_filtered, mean_expr, specificity = preprocess_for_sldsc(
    adata,
    celltype_col=celltype_key,
    gene_identifier_mode="ensembl",
    gene_col="gene",
    genome_build="GRCh37",
    inplace=False,
)
print(f"Specificity shape: {specificity.shape}")
specificity.head()

[2026-07-06 15:33:59,826] INFO:cellink.tl.external._sldsc_utils: Querying Ensembl BioMart (GRCh37)...


[2026-07-06 15:34:00,082] INFO:cellink.tl.external._sldsc_utils: Fetching gene annotations from GRCh37...


[2026-07-06 15:34:00,381] INFO:cellink.tl.external._sldsc_utils: Fetched annotations for 86413 genes from GRCh37


[2026-07-06 15:34:00,390] INFO:cellink.tl.external._sldsc_utils: Removing version suffixes from Gene IDs


[2026-07-06 15:34:00,404] INFO:cellink.tl.external._sldsc_utils: Dropping conflicting columns from adata.var before merge: ['chrom', 'start', 'end']


...

[2026-07-06 15:34:07,513] INFO:cellink.tl.external._sldsc_utils: Final data shape: (30000, 15288)


[2026-07-06 15:34:07,514] INFO:cellink.tl.external._sldsc_utils: Mean expression shape: (15288, 31)


[2026-07-06 15:34:07,516] INFO:cellink.tl.external._sldsc_utils: Specificity shape: (15288, 31)


Specificity shape: (15288, 31)


ClusterID,ASDC,B intermediate,B memory,B naive,CD14 Mono,CD16 Mono,CD4 CTL,CD4 Naive,CD4 Proliferating,CD4 TCM,...,NK Proliferating,NK_CD56bright,Plasmablast,Platelet,Treg,cDC1,cDC2,dnT,gdT,pDC
gene,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.025544,0.019785,0.026290,0.021848,0.020670,0.026413,0.022485,0.029190,0.045058,0.031264,...,0.046357,0.026829,0.063601,0.021355,0.031400,0.084555,0.025703,0.048542,0.022933,0.019958
ENSG00000000457,0.000000,0.027082,0.029501,0.028782,0.021906,0.022242,0.039849,0.029089,0.027443,0.030250,...,0.016565,0.020840,0.031538,0.000000,0.031054,0.101089,0.015698,0.058034,0.033522,0.034678
ENSG00000000460,0.000000,0.015683,0.007674,0.006417,0.004274,0.031242,0.014826,0.008407,0.385473,0.008488,...,0.025853,0.010842,0.070428,0.070675,0.004039,0.000000,0.008167,0.000000,0.004360,0.000000
ENSG00000000938,0.022795,0.042297,0.016634,0.023277,0.117960,0.149480,0.024831,0.002938,0.000000,0.003036,...,0.095511,0.073427,0.017761,0.030741,0.002107,0.042541,0.093090,0.000000,0.058943,0.003562
ENSG00000000971,0.000000,0.000000,0.009799,0.000000,0.004093,0.000000,0.119119,0.002415,0.000000,0.048274,...,0.000000,0.000000,0.031425,0.135365,0.000000,0.000000,0.000000,0.000000,0.091856,0.000000


---
## Part 1 — MAGMA 

MAGMA provides a complementary enrichment analysis that does not require LD score files. The pipeline has three steps:

| Step | Function | Output |
|---|---|---|
| I — Annotate | `run_magma_annotate` | `.genes.annot` |
| II — Gene analysis | `run_magma_gene_analysis` | `.genes.raw` |
| III-a — Gene-set analysis | `run_magma_gsa` | `.gsa.out` |
| III-b — Gene property analysis | `run_magma_gpa` | `.gsa.out` |

Steps I and II only need to be run once per GWAS. Step III is run once per score set / method.

### Step I — Annotate SNPs to genes

In [6]:
# Derive MAGMA's SNP location file (SNP / CHR / BP) from real GWAS summary
# statistics — the same study (GCST004787, coronary artery disease) used in
# the companion LDSC notebook's heritability/genetic-correlation sections.
gwas_summary_statistic_path = get_gwas_catalog_study_summary_stats("GCST004787", genome_build="GRCh37", return_path=True)
gwas_df = pd.read_csv(gwas_summary_statistic_path, sep="\t")
gwas_df = gwas_df.dropna(subset=["variant_id", "chromosome", "base_pair_location", "p_value"])
# chromosome/base_pair_location load as float64 (NaN forces the column to
# float dtype) — cast to int before filtering/writing, otherwise "22.0" never
# matches "22" and MAGMA receives malformed CHR/BP columns.
gwas_df["chromosome"] = gwas_df["chromosome"].astype(int)
gwas_df["base_pair_location"] = gwas_df["base_pair_location"].astype(int)
gwas_df = gwas_df[gwas_df["chromosome"] == 22]   # restrict to chr22, matching G1000_BFILE
print(f"{len(gwas_df):,} chr22 SNPs with usable GWAS data")

os.makedirs("magma_results", exist_ok=True)
GWAS_SNP_LOC = "magma_results/gwas_snp_loc.txt"
gwas_df[["variant_id", "chromosome", "base_pair_location"]].rename(
    columns={"variant_id": "SNP", "chromosome": "CHR", "base_pair_location": "BP"}
).to_csv(GWAS_SNP_LOC, sep="\t", index=False)

annot_result = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    window_kb=0,   # gene body only; use e.g. 35 for ±35 kb
)
print("Annotation file:", annot_result["annot_file"])


[2026-07-06 15:34:07,598] INFO:root: Fetching https://www.ebi.ac.uk/gwas/rest/api/v2/studies/GCST004787


[2026-07-06 15:34:08,141] INFO:root: User requested GRCh37, searching for a build-specific file


[2026-07-06 15:34:08,852] INFO:root: Selected file matching requested build GRCh37: 28714975-GCST004787-EFO_0001645-Build37.f.tsv.gz


[2026-07-06 15:34:08,853] INFO:root: Using build-specific summary statistics from harmonised/ (build: GRCh37)


[2026-07-06 15:34:08,856] INFO:root: Downloading http://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST004001-GCST005000/GCST004787/harmonised/28714975-GCST004787-EFO_0001645-Build37.f.tsv.gz to /home/icb/lucas.arnoldt/cellink_data/GCST004787_summary_stats.tsv.gz


121,746 chr22 SNPs with usable GWAS data


[2026-07-06 15:35:46,789] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad


[2026-07-06 15:35:47,506] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--annotate
	--snp-loc magma_results/gwas_snp_loc.txt
	--gene-loc /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt
	--out magma_results/cad

Start time is 15:35:46, Monday 06 Jul 2026

Starting annotation...
Reading gene locations from file /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt... 
	78696 gene locations read from file
	chromosome  1: 7108 genes
	chromosome  2: 5687 genes
	chromosome  3: 4159 genes
	chromosome  4: 3577 genes
	chromosome  5: 3948 genes
	chromosome  6: 4232 genes
	chromosome  7: 3958 genes
	chromosome  8: 3265 genes
	chromosome  9: 3188 genes
	chromosome 10: 3271 genes
	chromosome 11: 4171 genes
	chromosome 12: 3862 genes
	chromosome 13: 2000 genes
	chromosome 14: 2795 genes
	chromosome 15: 2819 genes
	chromosome 16: 3186 genes
	chromosome 17: 3793 genes
	chromosome 18: 1652 genes
	chromosome 19: 3543 genes
	chromosome 20: 19

Annotation file: magma_results/cad.genes.annot


In [7]:
# Preview the command without running:
cmd_preview = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    run=False,
)
print(" ".join(cmd_preview["command"]))


magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad


### Step II — Gene-level association analysis

This step uses a PLINK LD reference panel to compute gene-level z-scores from the GWAS p-values. The output `.genes.raw` file is the input for both GSA and GPA.

In [8]:
# GWAS p-value file: tab-delimited, columns SNP / P / N. The real summary
# statistics already carry a per-SNP sample size (n_samples), so we pass
# that through directly instead of a single global N.
GWAS_PVAL_FILE = "magma_results/gwas_pval.txt"
gwas_pval_df = gwas_df[["variant_id", "p_value", "n_samples"]].copy()
gwas_pval_df["n_samples"] = gwas_pval_df["n_samples"].astype(int)
gwas_pval_df.rename(
    columns={"variant_id": "SNP", "p_value": "P", "n_samples": "N"}
).to_csv(GWAS_PVAL_FILE, sep="\t", index=False)

gene_result = run_magma_gene_analysis(
    bfile=G1000_BFILE,
    pval_file=GWAS_PVAL_FILE,
    gene_annot=annot_result["annot_file"],
    out_prefix="magma_results/cad",
    ncol="N",   # tells MAGMA to read per-SNP N from the file's "N" column
    magma_bin=MAGMA_BIN,
)
print("Gene results:", gene_result["gene_results"])


[2026-07-06 15:35:48,041] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --bfile /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22 --pval magma_results/gwas_pval.txt ncol=N --gene-annot magma_results/cad.genes.annot --out magma_results/cad


[2026-07-06 15:36:41,686] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--bfile /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22
	--pval magma_results/gwas_pval.txt ncol=N
	--gene-annot magma_results/cad.genes.annot
	--out magma_results/cad

Start time is 15:35:48, Monday 06 Jul 2026

Loading PLINK-format data...
Reading file /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.fam... 489 individuals read
Reading file /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.bim... 141123 SNPs read
Preparing file /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22.bed... 

Reading SNP p-values from file magma_results/gwas_pval.txt... 
	detected 3 variables in file
	using variable: SNP (SNP id)
	using variable: P (p-value)
	using variable: N (sample size; discarding SNPs with N < 50)
	read 121747 lines from file, containing valid SNP p-values for 109778 SNPs in 

Gene results: magma_results/cad.genes.raw


In [9]:
# If you already have a pre-computed .genes.raw (e.g. from the combined_pipeline)
# you can skip Steps I and II and point directly to it. We just computed one
# for real above, so we use that:
GENES_RAW = gene_result["gene_results"]


---
## Part 2 — MAGMA Gene-Set Analysis (GSA)

GSA tests whether the top-scoring genes in each cell type show stronger GWAS association than background. There are two paths to create the required GMT file:

* **From a scores DataFrame** — use `scores_to_gmt` (selects top-N% per cell type)
* **From existing LDSC `.GeneSet` files** — use `genesets_dir_to_entrez_gmt`

### Path A — From continuous scores

In [10]:
# specificity is a genes × cell-types DataFrame with ENSG IDs as the index
gmt_file = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/onek1k_specificity_top10.gmt",
    top_frac=0.10,          # top 10% genes per cell type
    ascending=False,         # ascending=True would select the bottom 10%
    set_name_prefix="onek1k_specificity",  # optional prefix for set names
)
print("GMT file:", gmt_file)

# Preview the first two lines
with open(gmt_file) as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split("\t")
        print(f"Set '{parts[0]}': {len(parts)-2} genes")
        if i == 1:
            break

[2026-07-06 15:36:41,930] INFO:cellink.tl.external._ldsc2magma: scores_to_gmt: wrote 31 gene sets (top 10%, 0 skipped) → magma_gsa/onek1k_specificity_top10.gmt


GMT file: magma_gsa/onek1k_specificity_top10.gmt
Set 'onek1k_specificity_ASDC': 1528 genes
Set 'onek1k_specificity_B_intermediate': 1528 genes


In [11]:
# If gene scores use gene symbols rather than ENSG IDs, provide a mapping:
# gene_map can be a path to a TSV with columns gene_name / ensg_id,
# or a pd.Series indexed by symbol with ENSG values.

# gmt_file = scores_to_gmt(
#     scores=scores_with_symbols,
#     out_file="magma_gsa/scores_mapped.gmt",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Path B — From existing binary LDSC `.GeneSet` files

In [12]:
# First generate the *.GeneSet files (one per cell type, top 10% specificity using method from )
from cellink.tl.external import generate_sldsc_genesets
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# Converts the *.GeneSet files produced by generate_sldsc_genesets into GMT.
# Gene IDs are passed through as-is (no Ensembl → Entrez conversion).
gmt_from_genesets = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets.gmt",
    include_control=False,
)
print("GMT from GeneSet files:", gmt_from_genesets)

[2026-07-06 15:36:41,953] INFO:cellink.tl.external._sldsc_utils: Writing gene sets to ldsc_genesets


[2026-07-06 15:36:41,979] INFO:cellink.tl.external._sldsc_utils: specificity_df index looks like Ensembl IDs; using them directly.


[2026-07-06 15:36:41,983] INFO:cellink.tl.external._sldsc_utils: Selecting top 1529 genes (10.0%) per cell type


[2026-07-06 15:36:42,842] INFO:cellink.tl.external._sldsc_utils: Wrote control gene set with 15288 genes


[2026-07-06 15:36:42,846] INFO:cellink.tl.external._sldsc_utils: Generated 31 cell-type-specific gene sets


[2026-07-06 15:36:42,953] INFO:cellink.tl.external._ldsc2magma: Wrote 31 gene sets to magma_gsa/ldsc_genesets.gmt (skipped 0)


GMT from GeneSet files: magma_gsa/ldsc_genesets.gmt


### Run GSA

In [13]:
gsa_result = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_onek1k_specificity",
    magma_bin=MAGMA_BIN,
)
print("GSA results:", gsa_result["results_file"])

# Load and display results
gsa_df = pd.read_csv(gsa_result["results_file"], sep=r"\s+", comment="#")
gsa_df.sort_values("P").head(10)

[2026-07-06 15:36:42,965] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/onek1k_specificity_top10.gmt --out magma_gsa/cad_onek1k_specificity


[2026-07-06 15:36:48,002] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--set-annot magma_gsa/onek1k_specificity_top10.gmt
	--out magma_gsa/cad_onek1k_specificity

Start time is 15:36:42, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-set annotation...
Reading file magma_gsa/onek1k_specificity_top10.gmt... 
	31 gene-set definitions read from file
	found 31 gene sets containing genes defined in genotype data (containing a total of 341 unique genes)
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 31 gene sets

Parsing model specifications...
Inverting gene-gene correlation matrix...
	processing block 1 of 1
                                                             

GSA results: magma_gsa/cad_onek1k_specificity.gsa.out


,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P,FULL_NAME
18,onek1k_specificity_ILC,SET,34,0.279340,0.043674,0.15182,0.033018,onek1k_specificity_ILC
27,onek1k_specificity_cDC2,SET,38,0.185390,0.030597,0.14700,0.103760,onek1k_specificity_cDC2
4,onek1k_specificity_CD14_Mono,SET,43,0.126950,0.022246,0.13102,0.166390,onek1k_specificity_CD14_Mono
0,onek1k_specificity_ASDC,SET,49,0.110700,0.020659,0.11891,0.176050,onek1k_specificity_ASDC
8,onek1k_specificity_CD4_Proli...,SET,38,0.085001,0.014029,0.13130,0.258760,onek1k_specificity_CD4_Proliferating
30,onek1k_specificity_pDC,SET,35,0.063124,0.010010,0.13042,0.314240,onek1k_specificity_pDC
15,onek1k_specificity_Doublet,SET,52,0.040318,0.007743,0.11083,0.358050,onek1k_specificity_Doublet
16,onek1k_specificity_Eryth,SET,31,0.040986,0.006126,0.13359,0.379520,onek1k_specificity_Eryth
22,onek1k_specificity_NK_CD56br...,SET,43,0.034799,0.006098,0.11666,0.382760,onek1k_specificity_NK_CD56bright
20,onek1k_specificity_NK,SET,44,0.034841,0.006174,0.12088,0.386610,onek1k_specificity_NK


In [14]:
# Extra MAGMA flags are forwarded via **kwargs, e.g. joint competitive test:
# run_magma_gsa(..., model="multi")

---
## Part 3 — MAGMA Gene Property Analysis (GPA)

GPA tests the linear relationship between continuous per-gene scores and GWAS gene z-scores. Unlike GSA it uses all genes with scores — no top-N threshold. The covariate file contains one column per cell type.

### Build the covariate file

In [15]:
covar_file = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity.covar",
    negate=False,   # set negate=True to test enrichment in low-score genes
)
print("Covariate file:", covar_file)

# Preview
pd.read_csv(covar_file, sep="\t", index_col=0).head()

[2026-07-06 15:36:49,276] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15288 genes × 31 cell types → magma_gpa/onek1k_specificity.covar


Covariate file: magma_gpa/onek1k_specificity.covar


,ASDC,B_intermediate,B_memory,B_naive,CD14_Mono,CD16_Mono,CD4_CTL,CD4_Naive,CD4_Proliferating,CD4_TCM,...,NK_Proliferating,NK_CD56bright,Plasmablast,Platelet,Treg,cDC1,cDC2,dnT,gdT,pDC
GENE,,,,,,,,,,,,,,,,,,,,,
ENSG00000000419,0.025544,0.019785,0.026290,0.021848,0.020670,0.026413,0.022485,0.029190,0.045058,0.031264,...,0.046357,0.026829,0.063601,0.021355,0.031400,0.084555,0.025703,0.048542,0.022933,0.019958
ENSG00000000457,0.000000,0.027082,0.029501,0.028782,0.021906,0.022242,0.039849,0.029089,0.027443,0.030250,...,0.016565,0.020840,0.031538,0.000000,0.031054,0.101089,0.015698,0.058034,0.033522,0.034678
ENSG00000000460,0.000000,0.015683,0.007674,0.006417,0.004274,0.031242,0.014826,0.008407,0.385473,0.008488,...,0.025853,0.010842,0.070428,0.070675,0.004039,0.000000,0.008167,0.000000,0.004360,0.000000
ENSG00000000938,0.022795,0.042297,0.016634,0.023277,0.117960,0.149480,0.024831,0.002938,0.000000,0.003036,...,0.095511,0.073427,0.017761,0.030741,0.002107,0.042541,0.093090,0.000000,0.058943,0.003562
ENSG00000000971,0.000000,0.000000,0.009799,0.000000,0.004093,0.000000,0.119119,0.002415,0.000000,0.048274,...,0.000000,0.000000,0.031425,0.135365,0.000000,0.000000,0.000000,0.000000,0.091856,0.000000


In [16]:
# Gene symbols → ENSG mapping works the same way as in scores_to_gmt:
# covar_file = scores_to_covar(
#     scores=scores_with_symbols,
#     out_file="magma_gpa/scores_mapped.covar",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Joint GPA (default)

All cell types are tested in a single MAGMA call. MAGMA may drop highly correlated covariates via its collinearity filter. Use univariate mode if this is a concern.

In [17]:
gpa_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_joint",
    magma_bin=MAGMA_BIN,
    univariate=False,
)
print("GPA results:", gpa_result["results_file"])

gpa_df = pd.read_csv(gpa_result["results_file"], sep=r"\s+", comment="#")
gpa_df.sort_values("P").head(10)

[2026-07-06 15:36:49,453] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity.covar --out magma_gpa/cad_onek1k_specificity_joint


[2026-07-06 15:36:49,813] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar magma_gpa/onek1k_specificity.covar
	--out magma_gpa/cad_onek1k_specificity_joint

Start time is 15:36:49, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-level covariates...
Reading file magma_gpa/onek1k_specificity.covar... 
	detected 31 variables in file (using all)
	found 31 valid gene covariates, for 371 genes defined in genotype data
Processing missing values...
	found 986 genes not present in all input files: removing these from analysis
	371 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 31 gene covariates

Parsing model specifications...
Inverting

GPA results: magma_gpa/cad_onek1k_specificity_joint.gsa.out


,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P
7,CD4_Naive,COVAR,371,-3.63490,-0.141670,1.39290,0.009477
19,MAIT,COVAR,371,-3.91290,-0.092978,1.84150,0.034343
24,Platelet,COVAR,371,0.68919,0.069859,0.37322,0.065707
5,CD16_Mono,COVAR,371,-1.54200,-0.081716,0.89115,0.084508
2,B_memory,COVAR,371,-1.03730,-0.066223,0.64754,0.110150
28,dnT,COVAR,371,-1.86700,-0.065780,1.26500,0.140930
1,B_intermediate,COVAR,371,-1.36090,-0.064703,0.93371,0.145920
6,CD4_CTL,COVAR,371,-2.51720,-0.064253,1.80660,0.164450
20,NK,COVAR,371,0.97609,0.066052,0.76955,0.205550
30,pDC,COVAR,371,-0.68559,-0.050638,0.57561,0.234480


### Univariate GPA

Each cell type is tested in an independent MAGMA call. Slower than joint mode but guarantees a result for every cell type — recommended when covariate columns are highly correlated (e.g. residual CV scores or negated scores).

In [18]:
gpa_univ_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_univariate",
    magma_bin=MAGMA_BIN,
    univariate=True,
)
print("Univariate GPA results:", gpa_univ_result["results_file"])

gpa_univ_df = pd.read_csv(gpa_univ_result["results_file"], sep=r"\s+", comment="#")
gpa_univ_df.sort_values("P").head(10)

[2026-07-06 15:36:49,962] INFO:cellink.tl.external._ldsc2magma: GPA univariate: 31 cell types


[2026-07-06 15:36:50,005] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar /tmp/tmphiopcjn3/ct.covar --out /tmp/tmphiopcjn3/ct_out


[2026-07-06 15:36:50,298] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar /tmp/tmphiopcjn3/ct.covar
	--out /tmp/tmphiopcjn3/ct_out

Start time is 15:36:50, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-level covariates...
Reading file /tmp/tmphiopcjn3/ct.covar... 
	detected 1 variables in file (using all)
	found 1 valid gene covariate, for 371 genes defined in genotype data
Processing missing values...
	found 986 genes not present in all input files: removing these from analysis
	371 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 1 gene covariate

Parsing model specifications...
Inverting gene-gene correlation matrix...
	proc

[2026-07-06 15:36:50,300] INFO:cellink.tl.external._ldsc2magma:   ASDC: done


[2026-07-06 15:36:50,354] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar /tmp/tmphiopcjn3/ct.covar --out /tmp/tmphiopcjn3/ct_out


...

[2026-07-06 15:37:00,662] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar /tmp/tmphiopcjn3/ct.covar
	--out /tmp/tmphiopcjn3/ct_out

Start time is 15:37:00, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-level covariates...
Reading file /tmp/tmphiopcjn3/ct.covar... 
	detected 1 variables in file (using all)
	found 1 valid gene covariate, for 371 genes defined in genotype data
Processing missing values...
	found 986 genes not present in all input files: removing these from analysis
	371 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 1 gene covariate

Parsing model specifications...
Inverting gene-gene correlation matrix...
	proc

[2026-07-06 15:37:00,664] INFO:cellink.tl.external._ldsc2magma:   pDC: done


[2026-07-06 15:37:00,699] INFO:cellink.tl.external._ldsc2magma: GPA univariate: wrote 31 cell types → magma_gpa/cad_onek1k_specificity_univariate.gsa.out


Univariate GPA results: magma_gpa/cad_onek1k_specificity_univariate.gsa.out


,VARIABLE,TYPE,NGENES,BETA,BETA_STD,SE,P,FULL_NAME
7,CD4_Naive,COVAR,371,-3.63490,-0.141670,1.39290,0.009477,CD4_Naive
19,MAIT,COVAR,371,-3.91290,-0.092978,1.84150,0.034343,MAIT
24,Platelet,COVAR,371,0.68919,0.069859,0.37322,0.065707,Platelet
5,CD16_Mono,COVAR,371,-1.54200,-0.081716,0.89115,0.084508,CD16_Mono
2,B_memory,COVAR,371,-1.03730,-0.066223,0.64754,0.110150,B_memory
28,dnT,COVAR,371,-1.86700,-0.065780,1.26500,0.140930,dnT
1,B_intermediate,COVAR,371,-1.36090,-0.064703,0.93371,0.145920,B_intermediate
6,CD4_CTL,COVAR,371,-2.51720,-0.064253,1.80660,0.164450,CD4_CTL
20,NK,COVAR,371,0.97609,0.066052,0.76955,0.205550,NK
30,pDC,COVAR,371,-0.68559,-0.050638,0.57561,0.234480,pDC


---
## Part 4 — LDSC → MAGMA: Full Workflows

The two complete workflows from LDSC score outputs to MAGMA results.

### Workflow A: LDSC binary genesets → MAGMA GSA

In [19]:
# 1. Generate LDSC gene sets (top 10% specificity per cell type)
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# 2. Convert .GeneSet → GMT (MAGMA format, ENSG IDs kept as-is)
gmt_ldsc = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets_workflow_a.gmt",
)

# 3. GSA
gsa_a = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_ldsc),
    out_prefix="magma_gsa/workflow_a_cad",
    magma_bin=MAGMA_BIN,
)

results_a = pd.read_csv(gsa_a["results_file"], sep=r"\s+", comment="#")
print(results_a.sort_values("P").head())

[2026-07-06 15:37:00,722] INFO:cellink.tl.external._sldsc_utils: Writing gene sets to ldsc_genesets


[2026-07-06 15:37:00,749] INFO:cellink.tl.external._sldsc_utils: specificity_df index looks like Ensembl IDs; using them directly.


[2026-07-06 15:37:00,768] INFO:cellink.tl.external._sldsc_utils: Selecting top 1529 genes (10.0%) per cell type


[2026-07-06 15:37:00,983] INFO:cellink.tl.external._sldsc_utils: Wrote control gene set with 15288 genes


[2026-07-06 15:37:00,986] INFO:cellink.tl.external._sldsc_utils: Generated 31 cell-type-specific gene sets


[2026-07-06 15:37:01,068] INFO:cellink.tl.external._ldsc2magma: Wrote 31 gene sets to magma_gsa/ldsc_genesets_workflow_a.gmt (skipped 0)


[2026-07-06 15:37:01,070] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/ldsc_genesets_workflow_a.gmt --out magma_gsa/workflow_a_cad


[2026-07-06 15:37:06,427] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--set-annot magma_gsa/ldsc_genesets_workflow_a.gmt
	--out magma_gsa/workflow_a_cad

Start time is 15:37:01, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-set annotation...
Reading file magma_gsa/ldsc_genesets_workflow_a.gmt... 
	31 gene-set definitions read from file
	found 31 gene sets containing genes defined in genotype data (containing a total of 341 unique genes)
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 31 gene sets

Parsing model specifications...
Inverting gene-gene correlation matrix...
	processing block 1 of 1
                                                                     

             VARIABLE TYPE  NGENES      BETA  BETA_STD       SE         P
18                ILC  SET      34  0.279340  0.043674  0.15182  0.033018
27               cDC2  SET      38  0.185390  0.030597  0.14700  0.103760
4           CD14_Mono  SET      43  0.126950  0.022246  0.13102  0.166390
0                ASDC  SET      49  0.110700  0.020659  0.11891  0.176050
8   CD4_Proliferating  SET      38  0.085001  0.014029  0.13130  0.258760


### Workflow B: Continuous scores → MAGMA GPA

In [20]:
# 1. Build covariate file directly from the specificity score matrix
covar_b = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity_workflow_b.covar",
)

# 2. GPA (joint mode)
gpa_b = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_b),
    out_prefix="magma_gpa/workflow_b_cad",
    magma_bin=MAGMA_BIN,
)

results_b = pd.read_csv(gpa_b["results_file"], sep=r"\s+", comment="#")
print(results_b.sort_values("P").head())

[2026-07-06 15:37:07,603] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15288 genes × 31 cell types → magma_gpa/onek1k_specificity_workflow_b.covar


[2026-07-06 15:37:07,606] INFO:cellink.tl.external._ldsc2magma: Running: magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity_workflow_b.covar --out magma_gpa/workflow_b_cad


[2026-07-06 15:37:07,986] INFO:cellink.tl.external._ldsc2magma: Welcome to MAGMA v1.10 (linux/s)
Using flags:
	--gene-results magma_results/cad.genes.raw
	--gene-covar magma_gpa/onek1k_specificity_workflow_b.covar
	--out magma_gpa/workflow_b_cad

Start time is 15:37:07, Monday 06 Jul 2026

Reading file magma_results/cad.genes.raw... 
	1357 genes read from file
Loading gene-level covariates...
Reading file magma_gpa/onek1k_specificity_workflow_b.covar... 
	detected 31 variables in file (using all)
	found 31 valid gene covariates, for 371 genes defined in genotype data
Processing missing values...
	found 986 genes not present in all input files: removing these from analysis
	371 genes remaining in analysis
Preparing variables for analysis...
	truncating Z-scores 3 points below zero or 6 standard deviations above the mean
	truncating covariate values more than 5 standard deviations from the mean
	total variables available for analysis: 31 gene covariates

Parsing model specifications...
I

     VARIABLE   TYPE  NGENES     BETA  BETA_STD       SE         P
7   CD4_Naive  COVAR     371 -3.63490 -0.141670  1.39290  0.009477
19       MAIT  COVAR     371 -3.91290 -0.092978  1.84150  0.034343
24   Platelet  COVAR     371  0.68919  0.069859  0.37322  0.065707
5   CD16_Mono  COVAR     371 -1.54200 -0.081716  0.89115  0.084508
2    B_memory  COVAR     371 -1.03730 -0.066223  0.64754  0.110150


---
## Part 5 — Tips and Common Patterns

### Previewing commands without running

All MAGMA functions accept `run=False` to return the command list without executing it. Useful for submitting to a cluster.

In [21]:
for fn_name, result in [
    ("run_magma_annotate",
     run_magma_annotate(snp_loc=GWAS_SNP_LOC, gene_loc=MAGMA_GENE_LOC,
                        out_prefix="magma_results/cad", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gene_analysis",
     run_magma_gene_analysis(bfile=G1000_BFILE, pval_file=GWAS_PVAL_FILE,
                             gene_annot="magma_results/cad.genes.annot",
                             out_prefix="magma_results/cad",
                             ncol="N", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gsa",
     run_magma_gsa(gene_results=GENES_RAW, set_annot=str(gmt_file),
                   out_prefix="magma_gsa/preview", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gpa",
     run_magma_gpa(gene_results=GENES_RAW, gene_covar=str(covar_file),
                   out_prefix="magma_gpa/preview", magma_bin=MAGMA_BIN, run=False)),
]:
    print(f"{fn_name}:\n  {' '.join(result['command'])}\n")

run_magma_annotate:
  magma_bin/magma --annotate --snp-loc magma_results/gwas_snp_loc.txt --gene-loc /home/icb/lucas.arnoldt/cellink_data/magma/gene_loc_GRCh37.txt --out magma_results/cad

run_magma_gene_analysis:
  magma_bin/magma --bfile /home/icb/lucas.arnoldt/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.22 --pval magma_results/gwas_pval.txt ncol=N --gene-annot magma_results/cad.genes.annot --out magma_results/cad

run_magma_gsa:
  magma_bin/magma --gene-results magma_results/cad.genes.raw --set-annot magma_gsa/onek1k_specificity_top10.gmt --out magma_gsa/preview

run_magma_gpa:
  magma_bin/magma --gene-results magma_results/cad.genes.raw --gene-covar magma_gpa/onek1k_specificity.covar --out magma_gpa/preview



### Negating scores to test low-score enrichment

Some metrics (e.g. coefficient of variation) are expected to be *low* in disease-relevant genes. Pass `negate=True` to `scores_to_covar`, or `ascending=True` to `scores_to_gmt`, to select/test genes with the *lowest* scores:

In [22]:
# GSA: bottom 10% genes (lowest CV)
gmt_bottom = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/specificity_bottom10.gmt",
    top_frac=0.10,
    ascending=True,   # select lowest-scoring genes
)

# GPA: negate scores so low-score genes get high covariate values
covar_neg = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/specificity_negated.covar",
    negate=True,
)

[2026-07-06 15:37:08,197] INFO:cellink.tl.external._ldsc2magma: scores_to_gmt: wrote 31 gene sets (bottom 10%, 0 skipped) → magma_gsa/specificity_bottom10.gmt


[2026-07-06 15:37:09,367] INFO:cellink.tl.external._ldsc2magma: scores_to_covar: wrote 15288 genes × 31 cell types → magma_gpa/specificity_negated.covar


### Passing extra MAGMA flags

Any MAGMA option not explicitly supported can be forwarded via `**kwargs`. Each `key=value` pair becomes `--key value` in the MAGMA call:

In [23]:
# Competitive gene-set test (MAGMA's "multi" model)
run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_competitive",
    magma_bin=MAGMA_BIN,
    model="multi",   # forwarded as --model multi
    run=False,       # preview only
)

{'command': ['magma_bin/magma',
  '--gene-results',
  'magma_results/cad.genes.raw',
  '--set-annot',
  'magma_gsa/onek1k_specificity_top10.gmt',
  '--out',
  'magma_gsa/cad_competitive',
  '--model',
  'multi']}

---
## Summary

This tutorial demonstrated all new functions added to `cellink.tl.external`:

| Function | Purpose |
|---|---|
| `scores_to_gmt` | Convert score DataFrame → MAGMA GMT (top-N% per cell type) |
| `scores_to_covar` | Convert score DataFrame → MAGMA `.covar` (all genes, continuous) |
| `genesets_dir_to_entrez_gmt` | Convert LDSC `.GeneSet` files → MAGMA GMT |
| `run_magma_annotate` | MAGMA Step I: SNP → gene annotation |
| `run_magma_gene_analysis` | MAGMA Step II: gene-level association analysis |
| `run_magma_gsa` | MAGMA Step III-a: gene-set analysis (binary) |
| `run_magma_gpa` | MAGMA Step III-b: gene property analysis (continuous), with optional univariate mode |

Key properties shared across all functions:

* **`run=False`** returns the command list without executing — useful for HPC job submission.
* **Gene ID handling** — `scores_to_gmt` and `scores_to_covar` accept a `gene_map` argument for symbol → ENSG translation; rows that cannot be mapped to ENSG are dropped.
* **Univariate GPA** — `run_magma_gpa(univariate=True)` tests each cell type independently, avoiding MAGMA's collinearity filter when scores are highly correlated.